In [19]:
# Untuk gambar dan array
import os
import numpy as np
import pandas as pd
import shutil
import matplotlib.pyplot as plt

# Untuk ECC encryption (ECIES)
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

import cv2

# Untuk deep learning
import tensorflow as tf

# Untuk hashing
import hashlib

In [20]:
NUM_IMAGES = 1000
IMAGE_PATH = "../dataset"

In [ ]:
data = pd.read_csv("results.csv", delimiter='|')
data = data.drop(' comment_number', axis=1)
data

In [ ]:
print("Jumlah duplikasi: ", data.duplicated().sum())

In [ ]:
data = data.drop_duplicates()
print("Jumlah duplikasi: ", data.duplicated().sum())

In [ ]:
data = data.groupby('image_name').first().reset_index()
data

In [ ]:
cover_dir = '../dataset/cover_images'
os.makedirs(cover_dir, exist_ok=True)  # Membuat direktori jika belum ada

# 3. Ambil 1000 nama file teratas
top_image_names = data['image_name'].head(1000).tolist()

# 4. Iterasi dan pindahkan file
for image_name in top_image_names:
    source_path = os.path.join(IMAGE_PATH, image_name)
    target_path = os.path.join(cover_dir, image_name)
    if os.path.exists(source_path):
        shutil.move(source_path, target_path)
        print(f"Memindahkan '{image_name}' ke '{cover_dir}'")
    else:
        print(f"File '{image_name}' tidak ditemukan di '{IMAGE_PATH}'")

print("Selesai memindahkan gambar.")

In [ ]:
# Generate ECC private key
def generate_ecc_keys():
    private_key = ec.generate_private_key(ec.BrainpoolP256R1())  # BrainpoolP256r1 can be used if library supports
    public_key = private_key.public_key()
    return private_key, public_key

# ECC Encryption (hybrid with AES)
def ecc_encrypt(public_key, plaintext: str):
    # Step 1: Generate ephemeral ECC key pair
    ephemeral_private_key = ec.generate_private_key(ec.BrainpoolP256R1())
    ephemeral_public_key = ephemeral_private_key.public_key()
    
    # Step 2: Generate shared secret
    shared_secret = ephemeral_private_key.exchange(ec.ECDH(), public_key)
    
    # Step 3: Derive AES key from shared secret
    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'handshake data',
    ).derive(shared_secret)
    
    # Step 4: Encrypt plaintext with AES-GCM
    iv = os.urandom(12)  # 96-bit IV
    encryptor = Cipher(
        algorithms.AES(derived_key),
        modes.GCM(iv)
    ).encryptor()
    
    ciphertext = encryptor.update(plaintext.encode()) + encryptor.finalize()
    
    return {
        'ephemeral_public_key': ephemeral_public_key,
        'iv': iv,
        'ciphertext': ciphertext,
        'tag': encryptor.tag
    }

# ECC Decryption
def ecc_decrypt(private_key, encrypted_data):
    ephemeral_public_key = encrypted_data['ephemeral_public_key']
    iv = encrypted_data['iv']
    ciphertext = encrypted_data['ciphertext']
    tag = encrypted_data['tag']
    
    # Generate shared secret
    shared_secret = private_key.exchange(ec.ECDH(), ephemeral_public_key)
    
    # Derive AES key
    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'handshake data',
    ).derive(shared_secret)
    
    # Decrypt ciphertext
    decryptor = Cipher(
        algorithms.AES(derived_key),
        modes.GCM(iv, tag)
    ).decryptor()
    
    plaintext = decryptor.update(ciphertext) + decryptor.finalize()
    return plaintext.decode()

# Convert ciphertext to image
def ciphertext_to_image(ciphertext: bytes, image_size=(64, 64)):
    # Pad ciphertext to match image size
    needed_size = image_size[0] * image_size[1]
    
    # Convert ciphertext bytes to integer array
    int_cipher = list(ciphertext)
    
    # Padding if needed
    if len(int_cipher) < needed_size:
        int_cipher += [0] * (needed_size - len(int_cipher))
    else:
        int_cipher = int_cipher[:needed_size]
    
    # Convert to numpy array
    array = np.array(int_cipher, dtype=np.uint8)
    image = array.reshape(image_size)
    
    return image

# Save image (optional)
def save_image(image_array, filename):
    cv2.imwrite(filename, image_array)

In [ ]:
data_comments = data[' comment'].head(1000)

In [ ]:
# Generate ECC keys (1 pasang saja untuk semua comment)
private_key, public_key = generate_ecc_keys()

# Simpan semua secret images ke list
secret_images = []

# Loop semua comment
for idx, comment in enumerate(data_comments):
    # Encrypt comment
    encrypted_data = ecc_encrypt(public_key, comment)
    
    # Convert ciphertext to secret image
    secret_image = ciphertext_to_image(encrypted_data['ciphertext'], image_size=(64, 64))
    
    # Simpan ke list
    secret_images.append(secret_image)
    
    # (Opsional) Simpan sebagai file gambar
    secret_dir = '../dataset/secret_images'
    os.makedirs(secret_dir, exist_ok=True)
    save_image(secret_image, f'../dataset/secret_images/secret_image_{idx}.png')

print(f"Total secret images generated: {len(secret_images)}")

In [21]:
def build_preparation_network():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3), name='prep_secret_input'),
        tf.keras.layers.Conv2D(65, (3, 3), padding='same', activation='relu', name='prep_secret_conv1'),
        tf.keras.layers.Conv2D(65, (3, 3), padding='same', activation='relu', name='prep_secret_conv2'),
        tf.keras.layers.Conv2D(3, (3, 3), padding='same', activation='relu', name='prep_secret_output'),
    ], name='Preparation_Network')
    return model

# Build model
prep_net = build_preparation_network()

# Cek model summary
prep_net.summary()

Model: "Preparation_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ prep_secret_conv1 (Conv2D)      │ (None, 64, 64, 65)     │         1,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prep_secret_conv2 (Conv2D)      │ (None, 64, 64, 65)     │        38,090 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prep_secret_output (Conv2D)     │ (None, 64, 64, 3)      │         1,758 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,668 (162.77 KB)

 Trainable params: 41,668 (162.77 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
def build_hiding_network():
    input_cover = tf.keras.layers.Input(shape=(64, 64, 3), name="hiding_cover_input")
    input_secret = tf.keras.layers.Input(shape=(64, 64, 3), name="hiding_secret_input")
    combined_input = tf.keras.layers.Concatenate(axis=-1, name="hiding_combined_input")([input_cover, input_secret])
    conv1 = tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="hiding_conv1")(combined_input)
    conv2 = tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="hiding_conv2")(conv1)
    conv3 = tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="hiding_conv3")(conv2)
    conv4 = tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="hiding_conv4")(conv3)
    conv5 = tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="hiding_conv5")(conv4)
    conv_output = tf.keras.layers.Conv2D(3, (3, 3), activation='relu', padding='same', name="hiding_output")(conv5)
    
    model = tf.keras.models.Model(inputs=[input_cover, input_secret], outputs=conv_output, name="Hiding_Network")
    
    return model

# Build model
hiding_net = build_hiding_network()

# Cek model summary
hiding_net.summary()

Model: "Hiding_Network"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ hiding_cover_input  │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_secret_input │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_combined_in… │ (None, 64, 64, 6) │          0 │ hiding_cover_inp… │
│ (Concatenate)       │                   │            │ hiding_secret_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_conv1        │ (None, 64, 64,    │      3,575 │ hiding_combined_… │
│ (Conv2D)            │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_conv2        │ (None, 64, 64,    │     38,090 │ hiding_conv1[0][… │
│ (Conv2D)            │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_conv3        │ (None, 64, 64,    │     38,090 │ hiding_conv2[0][… │
│ (Conv2D)            │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_conv4        │ (None, 64, 64,    │     38,090 │ hiding_conv3[0][… │
│ (Conv2D)            │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_conv5        │ (None, 64, 64,    │     38,090 │ hiding_conv4[0][… │
│ (Conv2D)            │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hiding_output       │ (None, 64, 64, 3) │      1,758 │ hiding_conv5[0][… │
│ (Conv2D)            │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 157,693 (615.99 KB)

 Trainable params: 157,693 (615.99 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
def build_reveal_network():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3), name="reveal_container_input"),
        tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="reveal_conv1"),
        tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="reveal_conv2"),
        tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="reveal_conv3"),
        tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="reveal_conv4"),
        tf.keras.layers.Conv2D(65, (3, 3), activation='relu', padding='same', name="reveal_conv5"),
        tf.keras.layers.Conv2D(3, (3, 3), activation='relu', padding='same', name="reveal_output"),
    ], name="Reveal_Network")
    
    return model

# Build model
reveal_net  = build_reveal_network()

# Cek model summary
reveal_net .summary()

Model: "Reveal_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reveal_conv1 (Conv2D)           │ (None, 64, 64, 65)     │         1,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reveal_conv2 (Conv2D)           │ (None, 64, 64, 65)     │        38,090 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reveal_conv3 (Conv2D)           │ (None, 64, 64, 65)     │        38,090 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reveal_conv4 (Conv2D)           │ (None, 64, 64, 65)     │        38,090 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reveal_conv5 (Conv2D)           │ (None, 64, 64, 65)     │        38,090 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reveal_output (Conv2D)          │ (None, 64, 64, 3)      │         1,758 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 155,938 (609.13 KB)

 Trainable params: 155,938 (609.13 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
def build_steganography_model(prep_net, hiding_net, reveal_net):
    cover_input = tf.keras.layers.Input(shape=(64, 64, 3))
    secret_input = tf.keras.layers.Input(shape=(64, 64, 3))
    
    prepared_secret_1 = prep_net(secret_input)
    container_image_1 = hiding_net([cover_input, prepared_secret_1])
    
    prepared_secret_2 = prep_net(container_image_1)
    final_container_image = hiding_net([cover_input, prepared_secret_2])
    
    recovered_secret = reveal_net(final_container_image)
    
    model = tf.keras.models.Model(inputs=[cover_input, secret_input], outputs=[final_container_image, recovered_secret])
    return model

steganography_model = build_steganography_model(prep_net, hiding_net, reveal_net)
steganography_model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Preparation_Network │ (None, 64, 64, 3) │     41,668 │ input_layer_3[0]… │
│ (Sequential)        │                   │            │ Hiding_Network[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Hiding_Network      │ (None, 64, 64, 3) │    157,693 │ input_layer_2[0]… │
│ (Functional)        │                   │            │ Preparation_Netw… │
│                     │                   │            │ input_layer_2[0]… │
│                     │                   │            │ Preparation_Netw… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Reveal_Network      │ (None, 64, 64, 3) │    155,938 │ Hiding_Network[1… │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 355,299 (1.36 MB)

 Trainable params: 355,299 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
def steganography_loss(cover_image, container_image, secret_image, recovered_secret, beta=1.0):
    container_loss = tf.reduce_mean(tf.square(cover_image - container_image))
    secret_loss = tf.reduce_mean(tf.square(secret_image - recovered_secret))
    total_loss = container_loss + beta * secret_loss
    return total_loss, container_loss, secret_loss

In [26]:
def load_images_from_folder(folder_path, target_size=(64, 64), color_mode='rgb'):
    images = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
            img_path = os.path.join(folder_path, filename)
            if color_mode == 'grayscale':
                img = tf.keras.preprocessing.image.load_img(img_path, color_mode='grayscale', target_size=target_size)
            else:
                img = tf.keras.preprocessing.image.load_img(img_path, target_size=target_size)
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            images.append(img_array)
    return np.array(images)

# Load cover images (64x64 RGB)
cover_images = load_images_from_folder("../dataset/cover_images", target_size=(64, 64), color_mode='rgb')

# Load secret images (64x64 Grayscale)
secret_images = load_images_from_folder("../dataset/secret_images", target_size=(64, 64), color_mode='rgb')

cover_images = cover_images / 255.0
secret_images = secret_images / 255.0

print(f"Total cover images loaded: {cover_images.shape}")
print(f"Total secret images loaded: {secret_images.shape}")

dataset = {
    'cover': cover_images,
    'secret': secret_images
}

Total cover images loaded: (1000, 64, 64, 3)
Total secret images loaded: (1000, 64, 64, 3)


In [27]:
optimizer = tf.keras.optimizers.Adam()

def train_step(cover_image, secret_image, model, optimizer, beta=1.0):
    with tf.GradientTape() as tape:
        container_image, recovered_secret = model([cover_image, secret_image], training=True)
        total_loss, container_loss, secret_loss = steganography_loss(
            cover_image, container_image, secret_image, recovered_secret, beta
        )

    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return total_loss, container_loss, secret_loss

def train_model(dataset, model, optimizer, epochs=500, batch_size=32, beta=1.0):
    num_batches = len(dataset['cover']) // batch_size

    for epoch in range(epochs):
        total_loss = 0
        total_container_loss = 0
        total_secret_loss = 0

        for i in range(num_batches):
            batch_cover = dataset['cover'][i * batch_size:(i + 1) * batch_size]
            batch_secret = dataset['secret'][i * batch_size:(i + 1) * batch_size]

            batch_cover = tf.convert_to_tensor(batch_cover, dtype=tf.float32)
            batch_secret = tf.convert_to_tensor(batch_secret, dtype=tf.float32)

            loss, container_loss, secret_loss = train_step(batch_cover, batch_secret, model, optimizer, beta)

            total_loss += loss
            total_container_loss += container_loss
            total_secret_loss += secret_loss

        avg_loss = total_loss / num_batches
        avg_container_loss = total_container_loss / num_batches
        avg_secret_loss = total_secret_loss / num_batches

        print(f"Epoch {epoch+1}/{epochs}, Total Loss: {avg_loss.numpy():.6f}, "
            f"Container Loss: {avg_container_loss.numpy():.6f}, Secret Loss: {avg_secret_loss.numpy():.6f}")

        if (epoch + 1) % 5 == 0:
            model.save(f"steganography_model_epoch_{epoch + 1}.h5")

In [28]:
total_losses, container_losses, secret_losses = train_model(dataset, steganography_model, optimizer, epochs=500, batch_size=32)

KeyboardInterrupt: 

In [ ]:
# Plot 1: Total Loss
plt.figure(figsize=(10, 5))
plt.plot(total_losses, label='Total Loss')
plt.title('Total Loss selama Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot 2: Container Loss
plt.figure(figsize=(10, 5))
plt.plot(container_losses, label='Container Loss')
plt.title('Container Loss selama Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot 3: Secret Loss
plt.figure(figsize=(10, 5))
plt.plot(secret_losses, label='Total Loss')
plt.title('Total Loss selama Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()